# Camargo pipeline

Real-life: **ssd** trim. Synthetic: **none** trim. "First (full-trace)" runs HPO inline in the cell itself, both Real and Synthetic. Half-prefix/Plain-field reuse the already-trained model.

## Real

In [ ]:
import sys
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "steady_state_detection"))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "plain-field"))

REAL_DATA_DIR = ROOT / "data" / "real-life"
BEST_MODELS = ROOT / "best_models"
RESULTS = ROOT / "results"
TRIM_NAME = "ssd"
SSD_TRIM_CFG = {"method": "ssd", "frac": 0.6, "k": 1.5, "pct": 0.25}

REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]

print(f"ROOT = {ROOT}")
print(f"{len(REAL_DATASETS)} real-life datasets")


def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

### 1. First (full-trace)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from create_prefixes_from_windows import make_three_way_split
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
from camargo.trainer import CamargoTrainer
from camargo.params import default_params as camargo_params

CAMARGO_MAX_EVAL = 10


def pt_kpi_series(event_log, test_index_cc, test_index_tt, window="days"):
    pred_cc = create_concurrent_cases_timeseries(
        event_log, time_col="end_timestamp", case_col="caseid", window=window, plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        event_log, time_col="end_timestamp", case_col="caseid", window=window, plot=False)
    cc_arr = pred_cc.reindex(test_index_cc).ffill().bfill().fillna(0).to_numpy()
    tt_arr = pred_tt.reindex(test_index_tt).ffill().bfill().fillna(0).to_numpy()
    return cc_arr, tt_arr

camargo_results = {}
for name in REAL_DATASETS:
    run_name = f"{name}_test_full"
    metrics_path = RESULTS / "camargo_hpo" / TRIM_NAME / run_name / f"metrics_{run_name}.csv"

    print(f"\n{'='*60}\n{name} (camargo full-trace HPO)\n{'='*60}")
    if metrics_path.exists():
        print("  [skip] metrics already exist")
        continue

    df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(REAL_DATA_DIR / f"{name}.xes")

    _, _, val_as_test_df = make_three_way_split(
        df_full, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["train_split"], full_traces=True,
    )

    camargo_run  = f"{name}_{TRIM_NAME}_hpo"
    camargo_trim = TRIM_NAME
    params_c = camargo_params(camargo_run, max_eval=CAMARGO_MAX_EVAL, epochs=200)
    trainer_c = CamargoTrainer(
        train_df, val_df, test_df,
        camargo_run, params_c, trim_dir=camargo_trim,
        val_as_test_df=val_as_test_df, cc_val=cc["val"], tt_val=tt["val"],
    )
    trainer_c.preprocess()
    print(f"Activities: {len(trainer_c.ac_index)}  Roles: {len(trainer_c.rl_index)}")

    _model_on_disk = (
        trainer_c.best_model_path is not None
        and Path(trainer_c.best_model_path).exists()
    )
    if _model_on_disk:
        print(f"Model exists, skipping training (loss={trainer_c.best_loss:.4f})")
        camargo_train_s = 0.0
    else:
        if trainer_c.best_model_path is not None:
            print(f"  model path set but file missing ({trainer_c.best_model_path}) — will retrain")
            trainer_c.best_model_path = None
        t0 = time.perf_counter()
        trainer_c.train()
        camargo_train_s = round(time.perf_counter() - t0, 2)
        print(f"Train time : {camargo_train_s:.1f}s")
    print(f"Best loss  : {trainer_c.best_loss:.4f}")
    print(f"Best model : {trainer_c.best_model_path}")

    camargo_best_dir = BEST_MODELS / name / TRIM_NAME / "camargo"
    camargo_best_dir.mkdir(parents=True, exist_ok=True)
    summary = {
        "run_name":     camargo_run,
        "trim_dir":     camargo_trim,
        "best_loss":    trainer_c.best_loss,
        "train_time_s": camargo_train_s,
        "output_dir":   trainer_c.output_dir,
    }
    (camargo_best_dir / "hpo_summary.json").write_text(json.dumps(summary, indent=2))
    print(f"Saved -> {camargo_best_dir}/hpo_summary.json")

    pred_paths  = trainer_c.predict()
    event_log_c = trainer_c.to_event_log(pred_paths)
    cc_pred_c, tt_pred_c = pt_kpi_series(
        event_log_c, test_index_cc=cc["test"].index, test_index_tt=tt["test"].index)

    out_dir_c = metrics_path.parent
    out_dir_c.mkdir(parents=True, exist_ok=True)

    results_c = {
        "concurrent_cases": {"mse": mean_squared_error(cc["test"].to_numpy(), cc_pred_c),
                             "mae": mean_absolute_error(cc["test"].to_numpy(), cc_pred_c)},
        "throughput_time":  {"mse": mean_squared_error(tt["test"].to_numpy(), tt_pred_c),
                             "mae": mean_absolute_error(tt["test"].to_numpy(), tt_pred_c)},
    }
    camargo_results[name] = results_c
    pd.DataFrame([
        dict(dataset=run_name, series=s, model="camargo_hpo", mse=m["mse"], mae=m["mae"])
        for s, m in results_c.items()
    ]).to_csv(out_dir_c / f"metrics_{run_name}.csv", index=False)
    pd.DataFrame([
        dict(model="camargo_hpo", phase="predict", params_json="{}",
             val_mse=float("nan"), time_s=0.0, is_best=True, dataset=run_name, series="all"),
    ]).to_csv(out_dir_c / f"time_{run_name}.csv", index=False)

    for series_name, actual, pred, index in [
        ("concurrent_cases", cc["test"].to_numpy(), cc_pred_c, cc["test"].index),
        ("throughput_time",  tt["test"].to_numpy(), tt_pred_c, tt["test"].index),
    ]:
        m = results_c[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color="green",  label="actual",             linewidth=1.5)
        ax.plot(index, pred,   color="purple", label="Camargo-HPO (pred)", linestyle="--")
        ax.set_title(f"{run_name} — {series_name}  MSE={m['mse']:.4f}  MAE={m['mae']:.4f}")
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir_c / f"{series_name}.png", dpi=150, bbox_inches="tight")
        plt.show(); plt.close()
    print(f"Metrics saved -> {out_dir_c}")


### 2. Half-prefix

In [ ]:
from run_predictions_real import _run_camargo

for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (camargo half-prefix)\n{'='*60}")
    run_name = f"{name}_test_full"
    summary_path = BEST_MODELS / name / TRIM_NAME / "camargo" / "hpo_summary.json"
    if not summary_path.exists():
        print(f"  [skip] no trained model at {summary_path}")
        continue

    half_metrics = RESULTS / "camargo_hpo_half" / TRIM_NAME / run_name / f"metrics_{run_name}.csv"
    if half_metrics.exists():
        print("  [skip] half-prefix metrics already exist")
        continue

    df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(REAL_DATA_DIR / f"{name}.xes")
    _run_camargo("half", summary_path, run_name, TRIM_NAME, train_df, val_df, test_df, df_full, cc, tt)

### 3. Plain-field

In [ ]:
from runner import (
    get_inflight_cases, build_sos_cases, predict_camargo_plain_field,
    compute_cc_tt_metrics, save_pf_metrics, save_pf_plot,
)
from sos import most_frequent_first_activity, most_frequent_first_resource, empirical_arrival_hour_sampler
from arrival import compute_arrival_series, ProphetArrivalModel
from camargo.params import default_params as camargo_dp
from camargo.trainer import CamargoTrainer

for name in REAL_DATASETS:
    run_name = f"{name}_test_full"
    print(f"\n{'='*60}\n{name} (camargo plain-field)\n{'='*60}")

    pf_metrics = RESULTS / "plain_field" / "camargo" / TRIM_NAME / run_name / f"metrics_{run_name}.csv"
    pf_raw_dir = BEST_MODELS / name / TRIM_NAME / "camargo" / "pf"
    if pf_metrics.exists() and (pf_raw_dir / "event_log.csv").exists():
        print("  [skip] plain-field metrics + raw predictions already exist")
        continue

    summary_path = BEST_MODELS / name / TRIM_NAME / "camargo" / "hpo_summary.json"
    if not summary_path.exists():
        print(f"  [skip] no trained model at {summary_path}")
        continue

    df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(REAL_DATA_DIR / f"{name}.xes")
    val_split = cc["val_split"]
    inflight_df = get_inflight_cases(df_full, val_split, case_col="caseid", time_col="end_timestamp")

    known_df = pd.concat([train_df, val_df], ignore_index=True)
    val_split_ts = pd.Timestamp(val_split)
    if val_split_ts.tzinfo:
        val_split_ts = val_split_ts.tz_convert(None)
    arrivals = compute_arrival_series(df_full, case_col="caseid", time_col="end_timestamp")
    arrivals = arrivals[arrivals.index < val_split_ts]
    arrival_model = ProphetArrivalModel().fit(arrivals)
    predicted_arrivals = arrival_model.predict(pd.DatetimeIndex(cc["test"].index).tz_localize(None))
    sos_df = build_sos_cases(
        predicted_arrivals,
        most_frequent_first_activity(known_df), most_frequent_first_resource(known_df),
        empirical_arrival_hour_sampler(known_df),
    )

    cs = json.loads(summary_path.read_text())
    trainer_c = CamargoTrainer.from_saved(train_df, val_df, test_df, cs["run_name"],
                                          camargo_dp(cs["run_name"], epochs=50), trim_dir=cs["trim_dir"])
    pred_log = predict_camargo_plain_field(trainer_c, sos_df, inflight_df)

    pf_raw_dir.mkdir(parents=True, exist_ok=True)
    pred_log.to_csv(pf_raw_dir / "event_log.csv", index=False)

    cc_p, tt_p, m = compute_cc_tt_metrics(pred_log, cc["test"], tt["test"])
    save_pf_metrics(name, "camargo", m, pf_metrics.parent)
    save_pf_plot(cc_p, tt_p, cc["test"], tt["test"], pf_metrics.parent, name, TRIM_NAME, "camargo")
    print(f"  [camargo pf] cc_mae={m['cc_mae']:.2f}  (raw predictions saved -> {pf_raw_dir})")

## Synthetic

In [ ]:
import warnings, math
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from time_series_preprocessing import ts_splits_from_log
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
from create_prefixes_from_windows import load_event_log, make_three_way_split
from setttings import set_global_seed
from camargo.trainer import CamargoTrainer
from camargo.params  import default_params as camargo_params

_COLS = {
    'case:concept:name':    'caseid',
    'concept:name':         'task',
    'lifecycle:transition': 'event_type',
    'org:resource':         'user',
    'time:timestamp':       'end_timestamp',
}

def pt_kpi_series(event_log, test_index_cc, test_index_tt, window='days'):
    pred_cc = create_concurrent_cases_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    cc_arr = pred_cc.reindex(test_index_cc).ffill().bfill().fillna(0).to_numpy()
    tt_arr = pred_tt.reindex(test_index_tt).ffill().bfill().fillna(0).to_numpy()
    return cc_arr, tt_arr

set_global_seed(1904)

_EXCLUDE_DATASETS = {'loan_recency', 'o2c_recency'}
SYNTH_DATASETS = sorted(
    p for p in (ROOT / 'data' / 'synthetic').glob('*.xes')
    if p.stem not in _EXCLUDE_DATASETS
)
print(f'{len(SYNTH_DATASETS)} synthetic datasets: {[p.stem for p in SYNTH_DATASETS]}')


### 1. First (full-trace)

In [ ]:
CAMARGO_MAX_EVAL = 10

for _xes_path in SYNTH_DATASETS:
    DATASET_NAME = _xes_path.stem
    RUN_NAME = f'{DATASET_NAME}_test_full'
    print(f"\n{'='*60}\n{DATASET_NAME} (camargo synthetic HPO + first)\n{'='*60}")

    log = pm4py.read_xes(str(_xes_path))
    ts = ts_splits_from_log(
        log,
        trim_method=None, trim_pct=0.25, trim_k=1.5,
        trim_frac=0.60, trim_window=7,
        train_frac=0.7, val_frac=0.1,
        cut_date=None,
    )
    cc = ts['concurrent_cases']
    tt = ts['throughput_time']

    train_split = cc['train_split']
    val_split   = cc['val_split']

    df_raw = load_event_log(_xes_path, time_col='time:timestamp', case_col='case:concept:name')
    df = df_raw.rename(columns=_COLS)
    df['task'] = df['task'].fillna('unk')
    df['user'] = df['user'].fillna('unk') if 'user' in df.columns else 'unk'

    train_df, val_df, test_df = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=val_split, full_traces=True,
    )

    _, _, val_as_test_df = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=train_split, full_traces=True,
    )

    camargo_run  = f'{DATASET_NAME}_hpo'
    camargo_trim = 'hpo'
    params_c = camargo_params(camargo_run, max_eval=CAMARGO_MAX_EVAL, epochs=200)
    trainer_c = CamargoTrainer(
        train_df, val_df, test_df,
        camargo_run, params_c, trim_dir=camargo_trim,
        val_as_test_df=val_as_test_df, cc_val=cc['val'], tt_val=tt['val'],
    )
    trainer_c.preprocess()
    print(f'Activities: {len(trainer_c.ac_index)}  Roles: {len(trainer_c.rl_index)}')

    if trainer_c.best_model_path is not None:
        print('Model exists, skipping training')
        camargo_train_s = 0.0
    else:
        t0 = time.perf_counter()
        trainer_c.train()
        camargo_train_s = round(time.perf_counter() - t0, 2)
        print(f'Train time : {camargo_train_s:.1f}s')
    print(f'Best loss  : {trainer_c.best_loss:.4f}')
    print(f'Best model : {trainer_c.best_model_path}')

    camargo_best_dir = BEST_MODELS / DATASET_NAME / 'camargo'
    camargo_best_dir.mkdir(parents=True, exist_ok=True)
    summary = {
        'run_name':     camargo_run,
        'trim_dir':     camargo_trim,
        'best_loss':    trainer_c.best_loss,
        'train_time_s': camargo_train_s,
        'output_dir':   trainer_c.output_dir,
    }
    (camargo_best_dir / 'hpo_summary.json').write_text(json.dumps(summary, indent=2))
    print(f'Saved -> {camargo_best_dir}/hpo_summary.json')

    metrics_path = RESULTS / 'camargo_hpo' / 'none' / RUN_NAME / f'metrics_{RUN_NAME}.csv'
    if metrics_path.exists():
        print(f'  metrics already exist -> {metrics_path}')
        continue

    pred_paths  = trainer_c.predict()
    event_log_c = trainer_c.to_event_log(pred_paths)
    cc_pred_c, tt_pred_c = pt_kpi_series(
        event_log_c, test_index_cc=cc['test'].index, test_index_tt=tt['test'].index)

    out_dir_c = metrics_path.parent
    out_dir_c.mkdir(parents=True, exist_ok=True)

    results_c = {
        'concurrent_cases': {'mse': mean_squared_error(cc['test'].to_numpy(), cc_pred_c),
                             'mae': mean_absolute_error(cc['test'].to_numpy(), cc_pred_c)},
        'throughput_time':  {'mse': mean_squared_error(tt['test'].to_numpy(), tt_pred_c),
                             'mae': mean_absolute_error(tt['test'].to_numpy(), tt_pred_c)},
    }
    pd.DataFrame([
        dict(dataset=RUN_NAME, series=s, model='camargo_hpo', mse=m['mse'], mae=m['mae'])
        for s, m in results_c.items()
    ]).to_csv(out_dir_c / f'metrics_{RUN_NAME}.csv', index=False)

    for series_name, actual, pred, index in [
        ('concurrent_cases', cc['test'].to_numpy(), cc_pred_c, cc['test'].index),
        ('throughput_time',  tt['test'].to_numpy(), tt_pred_c, tt['test'].index),
    ]:
        m = results_c[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color='green',  label='actual',             linewidth=1.5)
        ax.plot(index, pred,   color='purple', label='Camargo-HPO (pred)', linestyle='--')
        ax.set_title(f'{RUN_NAME} — {series_name}  MSE={m["mse"]:.4f}  MAE={m["mae"]:.4f}')
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir_c / f'{series_name}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
    print(f'Metrics saved -> {out_dir_c}')


### 2. Half-prefix

In [ ]:
for _xes_path in SYNTH_DATASETS:
    DATASET_NAME = _xes_path.stem
    RUN_NAME = f'{DATASET_NAME}_test_full'

    metrics_path = RESULTS / 'camargo_hpo_half' / 'none' / RUN_NAME / f'metrics_{RUN_NAME}.csv'
    if metrics_path.exists():
        print(f'[skip] {DATASET_NAME}: half-prefix metrics already exist')
        continue

    summary_path = BEST_MODELS / DATASET_NAME / 'camargo' / 'hpo_summary.json'
    if not summary_path.exists():
        print(f'[skip] {DATASET_NAME}: no trained model')
        continue

    print(f"\n{'='*60}\n{DATASET_NAME} (camargo synthetic half-prefix)\n{'='*60}")

    log = pm4py.read_xes(str(_xes_path))
    ts = ts_splits_from_log(
        log,
        trim_method=None, trim_pct=0.25, trim_k=1.5,
        trim_frac=0.60, trim_window=7,
        train_frac=0.7, val_frac=0.1,
        cut_date=None,
    )
    cc = ts['concurrent_cases']
    tt = ts['throughput_time']

    df_raw = load_event_log(_xes_path, time_col='time:timestamp', case_col='case:concept:name')
    df = df_raw.rename(columns=_COLS)
    df['task'] = df['task'].fillna('unk')
    df['user'] = df['user'].fillna('unk')

    train_split = cc['train_split']
    val_split   = cc['val_split']
    train_df, val_df, test_df_std = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=val_split, full_traces=True,
    )

    _test_case_ids = set(test_df_std['caseid'].astype(str))
    _df_test_full  = (
        df[df['caseid'].astype(str).isin(_test_case_ids)]
        .copy()
        .sort_values(['caseid', 'end_timestamp'])
    )
    _half_parts = []
    for _cid, _grp in _df_test_full.groupby('caseid', sort=False):
        _n = len(_grp)
        _half_parts.append(_grp.iloc[: max(1, math.ceil(_n / 2))])
    test_df = pd.concat(_half_parts, ignore_index=True)

    cc_actual = cc['test'].to_numpy()
    tt_actual = tt['test'].to_numpy()

    summary_c   = json.loads(summary_path.read_text())
    _run_name_c = summary_c['run_name']
    _trim_dir_c = summary_c['trim_dir']

    params_c  = camargo_params(_run_name_c, max_eval=1, epochs=200)
    trainer_c = CamargoTrainer.from_saved(
        train_df, val_df, test_df,
        _run_name_c, params_c,
        trim_dir=_trim_dir_c,
    )

    _camargo_repo    = ROOT / 'GenerativeLSTM' / 'GenerativeLSTM'
    _camargo_el_half = _camargo_repo / trainer_c.output_dir / 'event_log_half_prefix.csv'

    if _camargo_el_half.exists():
        event_log_c = pd.read_csv(_camargo_el_half)
        predict_s_c = 0.0
        print('  loaded cached half-prefix event_log')
    else:
        t0 = time.perf_counter()
        pred_paths  = trainer_c.predict(full_prefix_only=True)
        event_log_c = trainer_c.to_event_log(pred_paths)
        predict_s_c = round(time.perf_counter() - t0, 2)
        event_log_c.to_csv(_camargo_el_half, index=False)

    cc_pred_c, tt_pred_c = pt_kpi_series(
        event_log_c, test_index_cc=cc['test'].index, test_index_tt=tt['test'].index)

    out_dir_c = metrics_path.parent
    out_dir_c.mkdir(parents=True, exist_ok=True)

    results_c = {
        'concurrent_cases': {'mse': mean_squared_error(cc_actual, cc_pred_c),
                             'mae': mean_absolute_error(cc_actual, cc_pred_c)},
        'throughput_time':  {'mse': mean_squared_error(tt_actual, tt_pred_c),
                             'mae': mean_absolute_error(tt_actual, tt_pred_c)},
    }
    pd.DataFrame([
        dict(dataset=RUN_NAME, series=s, model='camargo_hpo_half', mse=m['mse'], mae=m['mae'])
        for s, m in results_c.items()
    ]).to_csv(out_dir_c / f'metrics_{RUN_NAME}.csv', index=False)
    pd.DataFrame([
        dict(model='camargo_hpo_half', phase='predict', params_json='{}',
             val_mse=float('nan'), time_s=predict_s_c, is_best=True,
             dataset=RUN_NAME, series='all'),
    ]).to_csv(out_dir_c / f'time_{RUN_NAME}.csv', index=False)

    for series_name, actual, pred, index in [
        ('concurrent_cases', cc_actual, cc_pred_c, cc['test'].index),
        ('throughput_time',  tt_actual, tt_pred_c, tt['test'].index),
    ]:
        m = results_c[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color='green',  label='actual',                          linewidth=1.5)
        ax.plot(index, pred,   color='purple', label='Camargo-HPO half-prefix (pred)',  linestyle='--')
        ax.set_title(f'{RUN_NAME} — {series_name}  MSE={m["mse"]:.4f}  MAE={m["mae"]:.4f}  [half-prefix]')
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir_c / f'{series_name}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
    print(f'  {predict_s_c:.1f}s  ->  {out_dir_c}')


### 3. Plain-field

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "plain-field"))

from pf_lukas_prediction import run_pf_job, SYNTH_LOGS

for ds in SYNTH_LOGS:
    run_pf_job(ds, "none", is_real=False)
